# Unemployment rate - pull, clean, export

Indicator: unemployment rate, annual average of monthly (not seasonally adjusted) values.
Source: FRED. Geographies: Allegheny County, Pennsylvania, United States.

**API key:** set `FRED_API_KEY` as an environment variable - do not paste it into this notebook.

In [1]:
import json
import os
import time
from datetime import date
from pathlib import Path

import pandas as pd
import requests

API_KEY = os.environ.get("FRED_API_KEY")
if not API_KEY:
    raise RuntimeError("FRED_API_KEY not set. Set it as an environment variable, do not hardcode it here.")

RAW_DIR = Path("../raw")
DATA_DIR = Path("../data")
RAW_DIR.mkdir(exist_ok=True)
DATA_DIR.mkdir(exist_ok=True)

today = date.today().isoformat()
print("Pipeline run date:", today)

Pipeline run date: 2026-10-04


In [2]:
SERIES = {
    "Allegheny County": "PAALLE3URN",
    "Pennsylvania": "PAUR",
    "United States": "UNRATE",
}

YEARS = list(range(2014, 2024))  # edit as needed; FRED updates monthly

## 1. Pull

One request per geography - FRED returns the full monthly history for a series in one call, not
one call per year. Caches every raw response, dated, before any parsing.

In [3]:
raw_responses = {}
monthly_rows = []

for geo_name, series_id in SERIES.items():
    url = "https://api.stlouisfed.org/fred/series/observations"
    params = {"series_id": series_id, "api_key": API_KEY, "file_type": "json"}
    try:
        resp = requests.get(url, params=params, timeout=30)
        resp.raise_for_status()
        payload = resp.json()
    except Exception as exc:
        print(f"FAILED  geo={geo_name}  series={series_id}  error={exc}")
        continue

    raw_responses[geo_name] = payload

    for obs in payload.get("observations", []):
        year = int(obs["date"][:4])
        if year not in YEARS:
            continue
        value = None if obs["value"] == "." else float(obs["value"])
        monthly_rows.append({"year": year, "geography": geo_name, "value": value})

    time.sleep(0.2)

with open(RAW_DIR / f"unemployment_rate_raw_{today}.json", "w") as f:
    json.dump(raw_responses, f, indent=2)

monthly_df = pd.DataFrame(monthly_rows)
monthly_df.head()

,year,geography,value
0,2014,Allegheny County,6.1
1,2014,Allegheny County,6.2
2,2014,Allegheny County,6.0
3,2014,Allegheny County,5.0
4,2014,Allegheny County,5.5


## 2. Clean: annual average of monthly values

Standard way to compare county/state/national unemployment across years without a 12-point-per-year chart.

In [4]:
annual_df = (
    monthly_df.dropna(subset=["value"])
    .groupby(["year", "geography"], as_index=False)["value"]
    .mean()
    .round(1)
)

expected = len(YEARS) * len(SERIES)
print(f"Expected {expected} (year, geography) rows, got {len(annual_df)}")
annual_df

Expected 30 (year, geography) rows, got 30


,year,geography,value
0,2014,Allegheny County,5.4
1,2014,Pennsylvania,6.0
2,2014,United States,6.2
3,2015,Allegheny County,5.1
4,2015,Pennsylvania,5.4
5,2015,United States,5.3
6,2016,Allegheny County,5.0
7,2016,Pennsylvania,5.3
8,2016,United States,4.9
9,2017,Allegheny County,4.7


## 3. Export

In [5]:
wide = annual_df.pivot(index="year", columns="geography", values="value").reset_index()
series = wide.to_dict(orient="records")

payload = {
    "indicator": "Unemployment rate (%), annual average",
    "geographies": list(SERIES.keys()),
    "unit": "percent",
    "snapshot_as_of": today,
    "status": "live",
    "source": "Federal Reserve Economic Data (FRED), series " + ", ".join(SERIES.values()) + " (underlying data from the Bureau of Labor Statistics Local Area Unemployment Statistics program)",
    "series": series,
    "notes": "Annual average of monthly, not seasonally adjusted values.",
}

with open(DATA_DIR / "unemployment_rate.json", "w") as f:
    json.dump(payload, f, indent=2)

print("Wrote", DATA_DIR / "unemployment_rate.json")
payload

Wrote ../data/unemployment_rate.json


{'indicator': 'Unemployment rate (%), annual average',
 'geographies': ['Allegheny County', 'Pennsylvania', 'United States'],
 'unit': 'percent',
 'snapshot_as_of': '2026-10-04',
 'status': 'live',
 'source': 'Federal Reserve Economic Data (FRED), series PAALLE3URN, PAUR, UNRATE (underlying data from the Bureau of Labor Statistics Local Area Unemployment Statistics program)',
 'series': [{'year': 2014,
   'Allegheny County': 5.4,
   'Pennsylvania': 6.0,
   'United States': 6.2},
  {'year': 2015,
   'Allegheny County': 5.1,
   'Pennsylvania': 5.4,
   'United States': 5.3},
  {'year': 2016,
   'Allegheny County': 5.0,
   'Pennsylvania': 5.3,
   'United States': 4.9},
  {'year': 2017,
   'Allegheny County': 4.7,
   'Pennsylvania': 5.0,
   'United States': 4.4},
  {'year': 2018,
   'Allegheny County': 4.1,
   'Pennsylvania': 4.4,
   'United States': 3.9},
  {'year': 2019,
   'Allegheny County': 4.0,
   'Pennsylvania': 4.3,
   'United States': 3.7},
  {'year': 2020,
   'Allegheny County': 8